<a href="https://colab.research.google.com/github/divijdawar/CUDA/blob/main/CUDA_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import time
import torch
import torch.nn.functional as F

In [9]:
matrix = torch.randn(1024, 32768, device='cuda', dtype=torch.float32)

_ = torch.nn.functional.softmax(matrix, dim=-1)

torch.cuda.synchronize()

total_time = 0
n_iters = 5

for i in range(n_iters):
    torch.cuda.synchronize()
    start = time.time()
    _ = torch.nn.functional.softmax(matrix, dim=-1)
    torch.cuda.synchronize()
    end = time.time()

    total_time += (end - start) * 1000
    print(total_time)

print(f"Softmax computation time (average): {(total_time/n_iters):.3f} ms")

2.4657249450683594
4.656791687011719
6.839752197265625
9.015560150146484
11.194705963134766
Softmax computation time (average): 2.239 ms


In [12]:
%%writefile softmaxKernel.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <sys/time.h>

// Timing function
double get_time() {
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return tv.tv_sec + tv.tv_usec * 1e-6;
}

// Error checking macro
#define CUDA_CHECK(call) \
    do { \
        cudaError_t error = call; \
        if (error != cudaSuccess) { \
            printf("CUDA error at %s:%d: %s\n", __FILE__, __LINE__, \
                   cudaGetErrorString(error)); \
            exit(EXIT_FAILURE); \
        } \
    } while(0)

// Kernel definition
__global__ void softmaxKernel(float* input, float* output, int M, int N) {
    int row = blockDim.x * blockIdx.x + threadIdx.x;

    if (row < M) {
        // maximum of this row
        float x_max = -INFINITY;
        // norm factor of this row
        float norm = 0.0f;

        // First pass: find maximum
        for (int col = 0; col < N; col++) {
            int i = row * N + col;
            x_max = max(x_max, input[i]);
        }

        // Second pass: compute exponentials and sum
        for (int col = 0; col < N; col++) {
            int i = row * N + col;
            norm += expf(input[i] - x_max);
        }

        // Third pass: normalize
        for (int col = 0; col < N; col++) {
            int i = row * N + col;
            output[i] = expf(input[i] - x_max) / norm;
        }
    }
}

void initializeData(float* data, int size) {
    for (int i = 0; i < size; i++) {
        data[i] = (float)rand() / RAND_MAX;
    }
}

int main() {
    // Print CUDA device properties
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("\n=== Device Information ===\n");
    printf("Device Name: %s\n", prop.name);
    printf("Compute Capability: %d.%d\n", prop.major, prop.minor);
    printf("Max Threads per Block: %d\n", prop.maxThreadsPerBlock);
    printf("Max Thread Dimensions: (%d, %d, %d)\n",
           prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
    printf("Memory Clock Rate (MHz): %d\n", prop.memoryClockRate/1000);
    printf("Memory Bus Width (bits): %d\n", prop.memoryBusWidth);
    printf("Peak Memory Bandwidth (GB/s): %.1f\n\n",
           2.0*prop.memoryClockRate*(prop.memoryBusWidth/8)/1.0e6);

    // Problem dimensions
    const int M = 1024;  // Number of rows
    const int N = 256;   // Number of columns
    const int size = M * N * sizeof(float);

    // Performance metrics
    double start_time, end_time;
    double alloc_time, h2d_time, kernel_time, d2h_time, total_time;

    // Start timing
    start_time = get_time();

    // Host memory allocation
    float *h_input = (float*)malloc(size);
    float *h_output = (float*)malloc(size);
    initializeData(h_input, M * N);

    end_time = get_time();
    alloc_time = end_time - start_time;

    // Device memory allocation and H2D transfer
    start_time = get_time();
    float *d_input, *d_output;
    CUDA_CHECK(cudaMalloc(&d_input, size));
    CUDA_CHECK(cudaMalloc(&d_output, size));
    CUDA_CHECK(cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice));
    end_time = get_time();
    h2d_time = end_time - start_time;

    // Create CUDA events for kernel timing
    cudaEvent_t kernel_start, kernel_end;
    cudaEventCreate(&kernel_start);
    cudaEventCreate(&kernel_end);

    // Kernel execution
    int threadsPerBlock = 256;
    int blocksPerGrid = (M + threadsPerBlock - 1) / threadsPerBlock;

    // Warm-up run
    softmaxKernel<<<blocksPerGrid, threadsPerBlock>>>(d_input, d_output, M, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Timed run
    cudaEventRecord(kernel_start);
    softmaxKernel<<<blocksPerGrid, threadsPerBlock>>>(d_input, d_output, M, N);
    cudaEventRecord(kernel_end);
    cudaEventSynchronize(kernel_end);

    float kernel_milliseconds = 0;
    cudaEventElapsedTime(&kernel_milliseconds, kernel_start, kernel_end);
    kernel_time = kernel_milliseconds / 1000.0;

    // D2H transfer
    start_time = get_time();
    CUDA_CHECK(cudaMemcpy(h_output, d_output, size, cudaMemcpyDeviceToHost));
    end_time = get_time();
    d2h_time = end_time - start_time;

    // Calculate total time
    total_time = alloc_time + h2d_time + kernel_time + d2h_time;

    // Calculate FLOPS
    // Each row requires: N comparisons + 2N exp() + N divisions + N-1 additions
    // exp() typically takes ~20-30 cycles, let's count it as 25 FLOPS
    long long total_ops = M * (N + (2*N*25) + N + (N-1));
    double gflops = (total_ops / kernel_time) / 1e9;

    // Print performance metrics
    printf("\n=== Performance Metrics ===\n");
    printf("Data Size: %d x %d matrix (%0.2f MB)\n", M, N, size/(1024.0*1024.0));
    printf("Host Allocation Time: %f ms\n", alloc_time * 1000);
    printf("Host to Device Transfer Time: %f ms\n", h2d_time * 1000);
    printf("Kernel Execution Time: %f ms\n", kernel_time * 1000);
    printf("Device to Host Transfer Time: %f ms\n", d2h_time * 1000);
    printf("Total Time: %f ms\n", total_time * 1000);
    printf("Estimated GFLOPS: %0.2f\n", gflops);
    printf("Memory Bandwidth: %0.2f GB/s\n",
           (size * 2) / (h2d_time + d2h_time) / 1e9);

    // Verification
    bool correct = true;
    for (int row = 0; row < M; row++) {
        float sum = 0.0f;
        for (int col = 0; col < N; col++) {
            sum += h_output[row * N + col];
        }
        if (fabsf(sum - 1.0f) > 1e-5) {
            printf("Error in row %d: sum = %f\n", row, sum);
            correct = false;
            break;
        }
    }

    if (correct) {
        printf("\nSoftmax computation completed successfully!\n");
    }

    // Cleanup
    cudaEventDestroy(kernel_start);
    cudaEventDestroy(kernel_end);
    free(h_input);
    free(h_output);
    CUDA_CHECK(cudaFree(d_input));
    CUDA_CHECK(cudaFree(d_output));

    return 0;
}

Writing softmaxKernel.cu


In [13]:
!nvcc softmaxKernel.cu -o softmaxKernel

In [14]:
!./softmaxKernel


=== Device Information ===
Device Name: Tesla T4
Compute Capability: 7.5
Max Threads per Block: 1024
Max Thread Dimensions: (1024, 1024, 64)
Memory Clock Rate (MHz): 5001
Memory Bus Width (bits): 256
Peak Memory Bandwidth (GB/s): 320.1


=== Performance Metrics ===
Data Size: 1024 x 256 matrix (1.00 MB)
Host Allocation Time: 6.409168 ms
Host to Device Transfer Time: 174.752951 ms
Kernel Execution Time: 0.007456 ms
Device to Host Transfer Time: 0.925064 ms
Total Time: 182.094639 ms
Estimated GFLOPS: 1863.28
Memory Bandwidth: 0.01 GB/s
Error in row 0: sum = 0.000000


In [15]:
%%writefile onlineSoftmaxKernel.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <sys/time.h>

// Timing function
double get_time() {
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return tv.tv_sec + tv.tv_usec * 1e-6;
}

// Error checking macro
#define CUDA_CHECK(call) \
    do { \
        cudaError_t error = call; \
        if (error != cudaSuccess) { \
            printf("CUDA error at %s:%d: %s\n", __FILE__, __LINE__, \
                   cudaGetErrorString(error)); \
            exit(EXIT_FAILURE); \
        } \
    } while(0)

// Online Softmax Kernel
__global__ void onlineSoftmaxKernel(float* input, int M, int N) {
    int row = blockDim.x * blockIdx.x + threadIdx.x;

    if (row < M) {
        float x_max = -INFINITY;
        float norm = 0.0f;

        // pass 1: online max and norm computation
        for (int col = 0; col < N; col++) {
            int i = row * N + col;
            float curr = input[i];
            if (curr > x_max) {
                // correct the global norm here
                norm = norm * expf(x_max - curr);
                x_max = curr;
            }
            norm += expf(curr - x_max);
        }

        // pass 2: normalization
        for (int col = 0; col < N; col++) {
            int i = row * N + col;
            input[i] = expf(input[i] - x_max) / norm;
        }
    }
}

void initializeData(float* data, int size) {
    for (int i = 0; i < size; i++) {
        data[i] = (float)rand() / RAND_MAX;
    }
}

int main() {
    // Print CUDA device properties
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("\n=== Device Information ===\n");
    printf("Device Name: %s\n", prop.name);
    printf("Compute Capability: %d.%d\n", prop.major, prop.minor);
    printf("Max Threads per Block: %d\n", prop.maxThreadsPerBlock);
    printf("Memory Clock Rate (MHz): %d\n", prop.memoryClockRate/1000);
    printf("Memory Bus Width (bits): %d\n", prop.memoryBusWidth);
    printf("Peak Memory Bandwidth (GB/s): %.1f\n\n",
           2.0*prop.memoryClockRate*(prop.memoryBusWidth/8)/1.0e6);

    // Problem dimensions
    const int M = 1024;  // Number of rows
    const int N = 256;   // Number of columns
    const int size = M * N * sizeof(float);

    // Performance metrics
    double start_time, end_time;
    double alloc_time, h2d_time, kernel_time, d2h_time, total_time;

    // Start timing
    start_time = get_time();

    // Host memory allocation
    float *h_input = (float*)malloc(size);
    initializeData(h_input, M * N);

    end_time = get_time();
    alloc_time = end_time - start_time;

    // Device memory allocation and H2D transfer
    start_time = get_time();
    float *d_input;
    CUDA_CHECK(cudaMalloc(&d_input, size));
    CUDA_CHECK(cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice));
    end_time = get_time();
    h2d_time = end_time - start_time;

    // Create CUDA events for kernel timing
    cudaEvent_t kernel_start, kernel_end;
    cudaEventCreate(&kernel_start);
    cudaEventCreate(&kernel_end);

    // Kernel execution
    int threadsPerBlock = 256;
    int blocksPerGrid = (M + threadsPerBlock - 1) / threadsPerBlock;

    // Warm-up run
    onlineSoftmaxKernel<<<blocksPerGrid, threadsPerBlock>>>(d_input, M, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Timed run
    cudaEventRecord(kernel_start);
    onlineSoftmaxKernel<<<blocksPerGrid, threadsPerBlock>>>(d_input, M, N);
    cudaEventRecord(kernel_end);
    cudaEventSynchronize(kernel_end);

    float kernel_milliseconds = 0;
    cudaEventElapsedTime(&kernel_milliseconds, kernel_start, kernel_end);
    kernel_time = kernel_milliseconds / 1000.0;

    // D2H transfer
    start_time = get_time();
    CUDA_CHECK(cudaMemcpy(h_input, d_input, size, cudaMemcpyDeviceToHost));
    end_time = get_time();
    d2h_time = end_time - start_time;

    // Calculate total time
    total_time = alloc_time + h2d_time + kernel_time + d2h_time;

    // Calculate FLOPS
    // Each row requires: N comparisons + N exp() + N divisions + ~N/2 corrections
    // exp() typically takes ~20-30 cycles, let's count it as 25 FLOPS
    long long total_ops = M * (N + (N*25) + N + (N/2));
    double gflops = (total_ops / kernel_time) / 1e9;

    // Print performance metrics
    printf("\n=== Performance Metrics ===\n");
    printf("Data Size: %d x %d matrix (%0.2f MB)\n", M, N, size/(1024.0*1024.0));
    printf("Host Allocation Time: %f ms\n", alloc_time * 1000);
    printf("Host to Device Transfer Time: %f ms\n", h2d_time * 1000);
    printf("Kernel Execution Time: %f ms\n", kernel_time * 1000);
    printf("Device to Host Transfer Time: %f ms\n", d2h_time * 1000);
    printf("Total Time: %f ms\n", total_time * 1000);
    printf("Estimated GFLOPS: %0.2f\n", gflops);
    printf("Memory Bandwidth: %0.2f GB/s\n",
           (size * 2) / (h2d_time + d2h_time) / 1e9);

    // Verification
    bool correct = true;
    for (int row = 0; row < M; row++) {
        float sum = 0.0f;
        for (int col = 0; col < N; col++) {
            sum += h_input[row * N + col];
        }
        if (fabsf(sum - 1.0f) > 1e-5) {
            printf("Error in row %d: sum = %f\n", row, sum);
            correct = false;
            break;
        }
    }

    if (correct) {
        printf("\nOnline Softmax computation completed successfully!\n");
    }

    // Cleanup
    cudaEventDestroy(kernel_start);
    cudaEventDestroy(kernel_end);
    free(h_input);
    CUDA_CHECK(cudaFree(d_input));

    return 0;
}

Writing onlineSoftmaxKernel.cu


In [16]:
!nvcc onlineSoftmaxKernel.cu -o onlineSoftmaxKernel

In [17]:
!./onlineSoftmaxKernel


=== Device Information ===
Device Name: Tesla T4
Compute Capability: 7.5
Max Threads per Block: 1024
Memory Clock Rate (MHz): 5001
Memory Bus Width (bits): 256
Peak Memory Bandwidth (GB/s): 320.1


=== Performance Metrics ===
Data Size: 1024 x 256 matrix (1.00 MB)
Host Allocation Time: 7.418871 ms
Host to Device Transfer Time: 196.338177 ms
Kernel Execution Time: 0.005280 ms
Device to Host Transfer Time: 0.397921 ms
Total Time: 204.160248 ms
Estimated GFLOPS: 1365.33
Memory Bandwidth: 0.01 GB/s
Error in row 0: sum = 133.853165


In [22]:
%%writefile matmul_perf.cu
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// Utility function to check CUDA errors
#define CHECK_CUDA(call) { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        printf("CUDA Error: %s at line %d\n", cudaGetErrorString(err), __LINE__); \
        exit(1); \
    } \
}

// Custom kernel for matrix-vector multiplication
__global__ void matVecMulKernel(float* mat, float* vec, float* res, int M, int N) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M) {
        float sum = 0.0f;
        for (int col = 0; col < N; col++) {
            sum += mat[row * N + col] * vec[col];
        }
        res[row] = sum;
    }
}

int main() {
    // Problem dimensions
    const int M = 4096;  // Number of rows
    const int N = 4096;  // Number of columns
    const int NUM_RUNS = 100;  // Number of iterations for timing

    // Host memory allocation
    float *h_mat = (float*)malloc(M * N * sizeof(float));
    float *h_vec = (float*)malloc(N * sizeof(float));
    float *h_res = (float*)malloc(M * sizeof(float));
    float *h_res_cublas = (float*)malloc(M * sizeof(float));

    // Initialize data with random values
    srand(42);  // For reproducibility
    for (int i = 0; i < M * N; i++) h_mat[i] = (float)(rand()) / RAND_MAX;
    for (int i = 0; i < N; i++) h_vec[i] = (float)(rand()) / RAND_MAX;

    // Device memory allocation
    float *d_mat, *d_vec, *d_res;
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    printf("Starting performance measurements...\n");

    // Timing allocation
    cudaEventRecord(start);
    CHECK_CUDA(cudaMalloc(&d_mat, M * N * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&d_vec, N * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&d_res, M * sizeof(float)));
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float allocation_time;
    cudaEventElapsedTime(&allocation_time, start, stop);

    // Timing data transfer
    cudaEventRecord(start);
    CHECK_CUDA(cudaMemcpy(d_mat, h_mat, M * N * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_vec, h_vec, N * sizeof(float), cudaMemcpyHostToDevice));
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float transfer_time;
    cudaEventElapsedTime(&transfer_time, start, stop);

    // Custom kernel execution
    dim3 blockSize(256);
    dim3 gridSize((M + blockSize.x - 1) / blockSize.x);

    // Warmup run
    matVecMulKernel<<<gridSize, blockSize>>>(d_mat, d_vec, d_res, M, N);

    // Timing kernel execution
    cudaEventRecord(start);
    for (int i = 0; i < NUM_RUNS; i++) {
        matVecMulKernel<<<gridSize, blockSize>>>(d_mat, d_vec, d_res, M, N);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float kernel_time;
    cudaEventElapsedTime(&kernel_time, start, stop);
    kernel_time /= NUM_RUNS;  // Average time per run

    // cuBLAS execution
    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;

    // Warmup run
    cublasSgemv(handle, CUBLAS_OP_T, N, M, &alpha, d_mat, N, d_vec, 1, &beta, d_res, 1);

    // Timing cuBLAS execution
    cudaEventRecord(start);
    for (int i = 0; i < NUM_RUNS; i++) {
        cublasSgemv(handle, CUBLAS_OP_T, N, M, &alpha, d_mat, N, d_vec, 1, &beta, d_res, 1);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float cublas_time;
    cudaEventElapsedTime(&cublas_time, start, stop);
    cublas_time /= NUM_RUNS;  // Average time per run

    // Calculate performance metrics
    float gflops_kernel = (2.0f * M * N) / (kernel_time * 1e6);  // GFLOPS = (2*M*N ops) / (time in seconds)
    float gflops_cublas = (2.0f * M * N) / (cublas_time * 1e6);
    float bandwidth = (3.0f * sizeof(float) * M * N) / (kernel_time * 1e6);  // GB/s

    // Print performance metrics
    printf("\nPerformance Metrics:\n");
    printf("Matrix dimensions: %d x %d\n", M, N);
    printf("GPU Memory Allocation Time: %.3f ms\n", allocation_time);
    printf("Host-to-Device Transfer Time: %.3f ms\n", transfer_time);
    printf("Custom Kernel Execution Time: %.3f ms\n", kernel_time);
    printf("cuBLAS Execution Time: %.3f ms\n", cublas_time);
    printf("Custom Kernel Performance: %.2f GFLOPS\n", gflops_kernel);
    printf("cuBLAS Performance: %.2f GFLOPS\n", gflops_cublas);
    printf("Memory Bandwidth: %.2f GB/s\n", bandwidth);

    // Cleanup
    cudaFree(d_mat);
    cudaFree(d_vec);
    cudaFree(d_res);
    cublasDestroy(handle);
    free(h_mat);
    free(h_vec);
    free(h_res);
    free(h_res_cublas);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing matmul_perf.cu


In [23]:
!nvcc matmul_perf.cu -o matmul_perf -lcublas
!./matmul_perf

Starting performance measurements...

Performance Metrics:
Matrix dimensions: 4096 x 4096
GPU Memory Allocation Time: 0.274 ms
Host-to-Device Transfer Time: 17.725 ms
Custom Kernel Execution Time: 0.000 ms
cuBLAS Execution Time: 0.256 ms
Custom Kernel Performance: 85528.22 GFLOPS
cuBLAS Performance: 131.14 GFLOPS
Memory Bandwidth: 513169.31 GB/s


In [25]:
%%writefile matmul_perf_size.cu
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <stdio.h>
#include <stdlib.h>

#define CHECK_CUDA(call) { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        printf("CUDA Error: %s at line %d\n", cudaGetErrorString(err), __LINE__); \
        exit(1); \
    } \
}

__global__ void matVecMulKernel(float* mat, float* vec, float* res, int M, int N) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M) {
        float sum = 0.0f;
        for (int col = 0; col < N; col++) {
            sum += mat[row * N + col] * vec[col];
        }
        res[row] = sum;
    }
}

void benchmark_size(int M, int N) {
    const int NUM_RUNS = 100;

    // Host memory allocation and initialization
    float *h_mat = (float*)malloc(M * N * sizeof(float));
    float *h_vec = (float*)malloc(N * sizeof(float));
    float *h_res = (float*)malloc(M * sizeof(float));

    for (int i = 0; i < M * N; i++) h_mat[i] = (float)(rand()) / RAND_MAX;
    for (int i = 0; i < N; i++) h_vec[i] = (float)(rand()) / RAND_MAX;

    // Device memory allocation
    float *d_mat, *d_vec, *d_res;
    CHECK_CUDA(cudaMalloc(&d_mat, M * N * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&d_vec, N * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&d_res, M * sizeof(float)));

    CHECK_CUDA(cudaMemcpy(d_mat, h_mat, M * N * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_vec, h_vec, N * sizeof(float), cudaMemcpyHostToDevice));

    // Timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Custom kernel execution
    dim3 blockSize(256);
    dim3 gridSize((M + blockSize.x - 1) / blockSize.x);

    // Warmup
    matVecMulKernel<<<gridSize, blockSize>>>(d_mat, d_vec, d_res, M, N);

    cudaEventRecord(start);
    for (int i = 0; i < NUM_RUNS; i++) {
        matVecMulKernel<<<gridSize, blockSize>>>(d_mat, d_vec, d_res, M, N);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float kernel_time;
    cudaEventElapsedTime(&kernel_time, start, stop);
    kernel_time /= NUM_RUNS;

    // cuBLAS execution
    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;

    // Warmup
    cublasSgemv(handle, CUBLAS_OP_T, N, M, &alpha, d_mat, N, d_vec, 1, &beta, d_res, 1);

    cudaEventRecord(start);
    for (int i = 0; i < NUM_RUNS; i++) {
        cublasSgemv(handle, CUBLAS_OP_T, N, M, &alpha, d_mat, N, d_vec, 1, &beta, d_res, 1);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float cublas_time;
    cudaEventElapsedTime(&cublas_time, start, stop);
    cublas_time /= NUM_RUNS;

    // Calculate performance metrics
    float gflops_kernel = (2.0f * M * N) / (kernel_time * 1e6);
    float gflops_cublas = (2.0f * M * N) / (cublas_time * 1e6);
    float bandwidth = (3.0f * sizeof(float) * M * N) / (kernel_time * 1e6);

    printf("\nMatrix Size: %d x %d\n", M, N);
    printf("Custom Kernel Time: %.3f ms (%.2f GFLOPS)\n", kernel_time, gflops_kernel);
    printf("cuBLAS Time: %.3f ms (%.2f GFLOPS)\n", cublas_time, gflops_cublas);
    printf("Performance Ratio (cuBLAS/Custom): %.2fx\n", kernel_time/cublas_time);
    printf("Memory Bandwidth: %.2f GB/s\n", bandwidth);

    // Cleanup
    cudaFree(d_mat);
    cudaFree(d_vec);
    cudaFree(d_res);
    cublasDestroy(handle);
    free(h_mat);
    free(h_vec);
    free(h_res);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}

int main() {
    printf("Testing different matrix sizes...\n");

    // Test different sizes
    int sizes[] = {1024, 2048, 4096, 8192, 16384};
    for (int i = 0; i < 5; i++) {
        benchmark_size(sizes[i], sizes[i]);
    }

    return 0;
}

Writing matmul_perf_size.cu


In [28]:
!nvcc matmul_perf_size.cu -o matmul_perf_size -lcublas
!./matmul_perf_size

Testing different matrix sizes...

Matrix Size: 1024 x 1024
Custom Kernel Time: 0.000 ms (7886.40 GFLOPS)
cuBLAS Time: 0.021 ms (99.86 GFLOPS)
Performance Ratio (cuBLAS/Custom): 0.01x
Memory Bandwidth: 47318.41 GB/s

Matrix Size: 2048 x 2048
Custom Kernel Time: 0.000 ms (34492.63 GFLOPS)
cuBLAS Time: 0.074 ms (112.70 GFLOPS)
Performance Ratio (cuBLAS/Custom): 0.00x
Memory Bandwidth: 206955.78 GB/s

Matrix Size: 4096 x 4096
Custom Kernel Time: 0.000 ms (136002.06 GFLOPS)
cuBLAS Time: 0.256 ms (131.07 GFLOPS)
Performance Ratio (cuBLAS/Custom): 0.00x
Memory Bandwidth: 816012.44 GB/s

Matrix Size: 8192 x 8192
Custom Kernel Time: 0.000 ms (546133.31 GFLOPS)
cuBLAS Time: 1.008 ms (133.14 GFLOPS)
Performance Ratio (cuBLAS/Custom): 0.00x
Memory Bandwidth: 3276799.75 GB/s

Matrix Size: 16384 x 16384
Custom Kernel Time: 0.000 ms (2078961.00 GFLOPS)
cuBLAS Time: 3.976 ms (135.04 GFLOPS)
Performance Ratio (cuBLAS/Custom): 0.00x
Memory Bandwidth: 12473766.00 GB/s
